# Jour 1 · Mini-projet final — assistant helpdesk sourcé


![Architecture complète du mini-projet](../../assets/jour_01/05_architecture_finale.png)

*La ligne du haut prépare la base ; celle du bas répond aux questions en réutilisant cette base.*


## Critères de réussite

1. Les fichiers Markdown et PDF sont chargés.
2. Les documents sont découpés avec une taille et un overlap expliqués.
3. Les chunks possèdent des IDs stables.
4. Une relance ne multiplie pas les entrées.
5. La réponse utilise uniquement le contexte.
6. Les sources viennent des métadonnées.
7. Une question hors périmètre provoque un refus.
8. Au moins quatre cas évaluent le retrieval.


### Construire sans se perdre

Ne codez pas tout le projet en une seule fois. Terminez une étape, affichez un résultat simple, puis passez à la suivante : d'abord les chunks, ensuite les IDs, puis la recherche, le prompt et enfin la réponse.

À chaque étape, demandez-vous : « Quelle donnée entre ? Quelle donnée doit ressortir ? » Cette habitude rend le pipeline beaucoup plus facile à comprendre et à déboguer.


In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


## Étape A — charger et découper

Commencez par une fonction qui renvoie `list[Document]`. Affichez le nombre de documents, de chunks et quelques métadonnées.


In [ ]:
import hashlib

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def charger_et_decouper():
    docs = []
    for path in sorted(DOCUMENTS_DIR.glob("*.md")):
        if path.name == "README.md":
            continue
        loaded = TextLoader(str(path), encoding="utf-8").load()
        for doc in loaded:
            doc.metadata["source"] = path.name
        docs.extend(loaded)

    pdf_path = DOCUMENTS_DIR / "guide_escalade_p1.pdf"
    loaded_pdf = PyPDFLoader(str(pdf_path)).load()
    for doc in loaded_pdf:
        doc.metadata["source"] = pdf_path.name
    docs.extend(loaded_pdf)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=450,
        chunk_overlap=80,
        separators=["\n## ", "\n\n", "\n", " ", ""],
    )
    return splitter.split_documents(docs)


chunks_projet = charger_et_decouper()
print("Chunks :", len(chunks_projet))


## Étape B — indexer de façon idempotente

Créez un ID à partir de la source, de la page et du contenu. Utilisez une collection dans `outputs/chroma_projet`.


In [ ]:
from langchain_chroma import Chroma


def stable_id(chunk):
    raw = (
        f"{chunk.metadata.get('source')}|"
        f"{chunk.metadata.get('page', '')}|"
        f"{chunk.page_content}"
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def indexer(chunks):
    store = Chroma(
        collection_name="projet_helpdesk",
        embedding_function=embeddings,
        persist_directory=str(OUTPUTS_DIR / "chroma_projet"),
    )
    store.add_documents(chunks, ids=[stable_id(chunk) for chunk in chunks])
    return store


vectorstore_projet = indexer(chunks_projet)
print("Entrées indexées :", len(vectorstore_projet.get()["ids"]))


## Étape C — répondre avec les sources

La fonction doit retourner au minimum `reponse`, `sources` et `documents`.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt_projet = ChatPromptTemplate.from_messages([
    (
        "system",
        "Tu es un assistant helpdesk. Utilise uniquement le contexte. "
        "Si l'information manque, écris exactement : "
        "Je ne trouve pas cette information dans la documentation fournie.\n\n"
        "CONTEXTE :\n{contexte}",
    ),
    ("human", "{question}"),
])
generation_projet = prompt_projet | llm | StrOutputParser()


def repondre(question, k=3):
    docs = vectorstore_projet.similarity_search(question, k=k)
    contexte = "\n\n".join(
        f"[Source : {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    )
    texte = generation_projet.invoke({"contexte": contexte, "question": question})
    sources = sorted({doc.metadata.get("source") for doc in docs})
    return {"reponse": texte, "sources": sources, "documents": docs}


demo = repondre("Quel portail VPN faut-il saisir ?")
print(demo["reponse"])
print("Sources :", demo["sources"])


## Étape D — évaluer avant de conclure

Préparez des questions directes, reformulées et hors périmètre. Vérifiez d'abord les sources, puis les réponses.


In [ ]:
cas_projet = [
    ("Quel portail VPN faut-il saisir ?", "procedure_vpn.md"),
    ("Que vérifier si mon moniteur reste noir ?", "procedure_ecran.md"),
    ("Comment retirer une feuille bloquée ?", "procedure_imprimante.md"),
    ("Puis-je donner mon code MFA au support ?", "politique_securite.md"),
]

hits = []
for question, source_attendue in cas_projet:
    docs = vectorstore_projet.similarity_search(question, k=2)
    sources = [doc.metadata.get("source") for doc in docs]
    hit = source_attendue in sources
    hits.append(hit)
    print(question)
    print("→", sources, "OK" if hit else "À ANALYSER")

print(f"hit@2 : {sum(hits) / len(hits):.0%}")


## Présenter un échec utilement

Une erreur n'est pas seulement « le modèle s'est trompé ». Identifiez la première étape incorrecte.


![Ordre de diagnostic du pipeline RAG](../../assets/jour_01/05_ordre_debug.png)

*Question, retrieval, prompt puis réponse : on inspecte les étapes dans leur ordre réel.*


## Questions de synthèse

1. Pourquoi ne place-t-on pas un guide complet dans un seul chunk ?
2. Pourquoi un retriever peut-il renvoyer des documents pour une question hors sujet ?
3. Pourquoi les sources ne doivent-elles pas être générées librement ?
4. Quel problème les IDs stables résolvent-ils ?
5. Si la bonne source n'apparaît pas dans le top-k, que faut-il examiner ?

## Bilan de la journée

```text
fichiers
  → Documents
  → chunks
  → embeddings
  → Chroma
  → retriever
  → contexte
  → prompt
  → LLM
  → réponse + sources
```

La mémoire conversationnelle, les agents et l'interface viendront ensuite comme des couches séparées.
